# 01 -- Pillow avance

## Objectifs pedagogiques

A la fin de ce notebook, vous saurez :

- ouvrir, creer et sauvegarder des images avec Pillow ;
- redimensionner, rogner, tourner, transformer ;
- appliquer des filtres (flou, nettete, contours) ;
- manipuler pixels et canaux couleur ;
- dessiner formes et texte ;
- traiter par lots (thumbnails) ;
- gerer les modes couleur RGB / RGBA / L / P ;
- composer et creer des collages.

## Prerequis -- ce que vous devez maitriser

- fonctions, `pathlib`, context managers ;
- notions de pixel et de couleur.

Ce que ce notebook **ne couvrira pas** :

- OpenCV (voir `02_Opencv_intro.ipynb`) ;
- le deep learning sur images ;
- la video.

## Plan

1. Installation
2. Creer une image
3. Ouvrir et sauvegarder
4. Infos image
5. Redimensionnement
6. Rogner / crop
7. Rotations et transforms
8. Filtres
9. Pixels et canaux
10. Dessiner
11. Texte
12. Composition
13. Traitement par lots
14. Formats et optimisation
15. Synthese
16. Exercices

---

## 1. Installation

In [ ]:
# pip install pillow
from PIL import Image, ImageFilter, ImageDraw, ImageFont, ImageEnhance, ImageOps
import PIL
PIL.__version__

---

## 2. Creer une image

In [ ]:
img = Image.new('RGB', (200, 100), color=(70, 130, 180))
img

In [ ]:
# Degrade manuel
import numpy as np
arr = np.zeros((100, 200, 3), dtype=np.uint8)
arr[:, :, 0] = np.linspace(0, 255, 200, dtype=np.uint8)
Image.fromarray(arr)

---

## 3. Ouvrir et sauvegarder

In [ ]:
import tempfile, pathlib
tmp = pathlib.Path(tempfile.gettempdir())
img.save(tmp / 'demo.png')
reopen = Image.open(tmp / 'demo.png')
reopen

---

## 4. Infos image

In [ ]:
img = Image.new('RGB', (640, 480), 'steelblue')
print('size  :', img.size)
print('mode  :', img.mode)
print('format:', img.format)

### Modes couleur

| Mode | Description |
|---|---|
| `1` | binaire (1 bit) |
| `L` | niveaux de gris 8 bits |
| `P` | palette 8 bits |
| `RGB` | couleur 24 bits |
| `RGBA` | couleur + alpha |
| `CMYK` | impression |

---

## 5. Redimensionnement

In [ ]:
img = Image.new('RGB', (400, 300), 'tomato')
small = img.resize((100, 75))
small.size

In [ ]:
# Preserver le ratio
img.thumbnail((100, 100))  # in-place
img.size

In [ ]:
# Filtres d'interpolation
from PIL.Image import Resampling
img.resize((50, 50), Resampling.LANCZOS)

---

## 6. Rogner / crop

In [ ]:
img = Image.new('RGB', (400, 300), 'lightblue')
cropped = img.crop((50, 50, 350, 250))  # (left, top, right, bottom)
cropped.size

In [ ]:
# Crop centre
w, h = img.size
left = (w - 200) // 2
top = (h - 150) // 2
img.crop((left, top, left + 200, top + 150)).size

---

## 7. Rotations et transforms

In [ ]:
img = Image.new('RGB', (200, 100), 'orange')
img.rotate(45, expand=True)

In [ ]:
img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)

In [ ]:
img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)

---

## 8. Filtres

In [ ]:
img = Image.new('RGB', (200, 100), (100, 150, 200))
img.filter(ImageFilter.BLUR)

In [ ]:
img.filter(ImageFilter.GaussianBlur(radius=5))

In [ ]:
img.filter(ImageFilter.CONTOUR)

In [ ]:
img.filter(ImageFilter.SHARPEN)

In [ ]:
img.filter(ImageFilter.EDGE_ENHANCE)

---

## 9. Pixels et canaux

In [ ]:
img = Image.new('RGB', (4, 4), 'white')
img.putpixel((0, 0), (255, 0, 0))
img.getpixel((0, 0))

In [ ]:
# Canaux RGB
img = Image.new('RGB', (100, 100), (128, 64, 200))
r, g, b = img.split()
r.size, r.mode

In [ ]:
# Conversion
img.convert('L')  # niveaux de gris

---

## 10. Dessiner

In [ ]:
img = Image.new('RGB', (300, 200), 'white')
draw = ImageDraw.Draw(img)
draw.rectangle((10, 10, 100, 80), fill='red', outline='black')
draw.ellipse((120, 10, 220, 110), fill='blue')
draw.line((230, 10, 290, 120), fill='green', width=3)
img

---

## 11. Texte

In [ ]:
img = Image.new('RGB', (300, 80), 'white')
draw = ImageDraw.Draw(img)
try:
    font = ImageFont.truetype('DejaVuSans.ttf', 24)
except OSError:
    font = ImageFont.load_default()
draw.text((10, 25), 'Hello Pillow', fill='black', font=font)
img

---

## 12. Composition

In [ ]:
bg = Image.new('RGBA', (200, 200), (255, 255, 255, 255))
fg = Image.new('RGBA', (100, 100), (255, 0, 0, 128))
bg.paste(fg, (50, 50), fg)
bg

In [ ]:
# Alpha blend
from PIL import Image
a = Image.new('RGB', (100, 100), 'red')
b = Image.new('RGB', (100, 100), 'blue')
Image.blend(a, b, 0.5)

---

## 13. Traitement par lots

In [ ]:
import pathlib
import tempfile

# Creer quelques images
tmp = pathlib.Path(tempfile.mkdtemp())
for i in range(3):
    Image.new('RGB', (400, 300), (i * 80, 100, 200)).save(tmp / f'img{i}.png')

# Faire les thumbnails
for p in tmp.glob('*.png'):
    with Image.open(p) as im:
        im.thumbnail((100, 100))
        im.save(tmp / f'thumb_{p.name}')
print('Thumbnails crees dans', tmp)

---

## 14. Formats et optimisation

In [ ]:
img = Image.new('RGB', (500, 400), 'steelblue')
img.save(tmp / 'q80.jpg', quality=80, optimize=True)
img.save(tmp / 'q95.jpg', quality=95)
(tmp / 'q80.jpg').stat().st_size, (tmp / 'q95.jpg').stat().st_size

---

## Synthese

| Operation | Pillow |
|---|---|
| Creer | `Image.new(mode, size, color)` |
| Ouvrir | `Image.open(path)` |
| Sauver | `img.save(path, quality=)` |
| Resize | `img.resize(size, resample=)` |
| Thumbnail | `img.thumbnail(size)` (in-place) |
| Crop | `img.crop((l, t, r, b))` |
| Rotate | `img.rotate(angle, expand=True)` |
| Flip | `img.transpose(Transpose.FLIP_*)` |
| Filtre | `img.filter(ImageFilter.BLUR)` |
| Dessiner | `ImageDraw.Draw(img).rectangle(...)` |
| Texte | `draw.text((x, y), s, font=)` |
| Paste | `bg.paste(fg, pos, mask)` |

### Regles a retenir

- **`with Image.open(...)`** pour liberer la ressource.
- **`thumbnail` vs `resize`** : thumbnail preserve le ratio, in-place.
- **RGBA** pour la transparence.
- **`quality=85-90`** : bon compromis JPEG.

---

## Exercices

### Exercice 1 (facile) -- Creer un damier

Creez une image 200x200 avec un damier 10x10 (cases noires et blanches).

In [ ]:
from PIL import Image, ImageDraw

# TODO

<details>
<summary>Voir la correction</summary>

```python
from PIL import Image, ImageDraw

img = Image.new('RGB', (200, 200), 'white')
draw = ImageDraw.Draw(img)
for i in range(10):
    for j in range(10):
        if (i + j) % 2 == 0:
            draw.rectangle((i*20, j*20, (i+1)*20, (j+1)*20), fill='black')
img
```

</details>

### Exercice 2 (facile) -- Resize

Ecrivez une fonction `make_thumbnail(path, max_side)` qui cree une miniature en preservant le ratio.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from PIL import Image
import pathlib

def make_thumbnail(path: str, max_side: int) -> None:
    p = pathlib.Path(path)
    with Image.open(p) as im:
        im.thumbnail((max_side, max_side))
        im.save(p.with_stem(p.stem + '_thumb'))
```

</details>

### Exercice 3 (moyen) -- Watermark

Fonction `watermark(path, text)` qui ajoute un texte en bas a droite avec semi-transparence.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from PIL import Image, ImageDraw, ImageFont

def watermark(path: str, text: str) -> Image.Image:
    img = Image.open(path).convert('RGBA')
    overlay = Image.new('RGBA', img.size, (0, 0, 0, 0))
    draw = ImageDraw.Draw(overlay)
    font = ImageFont.load_default()
    w, h = img.size
    draw.text((w - 120, h - 25), text, fill=(255, 255, 255, 180), font=font)
    return Image.alpha_composite(img, overlay)
```

</details>

### Exercice 4 (moyen) -- Collage 2x2

Fonction `collage(images: list[Image.Image]) -> Image.Image` qui prend 4 images et les dispose en grille 2x2 (chacune 200x200).

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from PIL import Image

def collage(images: list[Image.Image]) -> Image.Image:
    assert len(images) == 4
    out = Image.new('RGB', (400, 400), 'white')
    for i, im in enumerate(images):
        resized = im.resize((200, 200))
        out.paste(resized, ((i % 2) * 200, (i // 2) * 200))
    return out
```

</details>

### Exercice 5 (difficile) -- Conversion couleur -> gris avec contraste

Ecrivez `enhance(path)` qui :
1. ouvre l'image ;
2. convertit en niveaux de gris ;
3. augmente le contraste de 50% ;
4. applique un filtre de nettete ;
5. sauvegarde en `*_enhanced.png`.

In [ ]:
# TODO

<details>
<summary>Voir la correction</summary>

```python
from PIL import Image, ImageEnhance, ImageFilter
import pathlib

def enhance(path: str) -> None:
    p = pathlib.Path(path)
    img = Image.open(p).convert('L')
    img = ImageEnhance.Contrast(img).enhance(1.5)
    img = img.filter(ImageFilter.SHARPEN)
    img.save(p.with_stem(p.stem + '_enhanced').with_suffix('.png'))
```

</details>

---

## Ressources externes

- [Pillow documentation](https://pillow.readthedocs.io/)
- [Image modes](https://pillow.readthedocs.io/en/stable/handbook/concepts.html#concept-modes)
- [ImageDraw](https://pillow.readthedocs.io/en/stable/reference/ImageDraw.html)